In [1]:
import sys
# Add your lenscarf site-packages to the path
sys.path.append('/users/odarwish/lenscarf/lib/python3.12/site-packages')

import numpy as np

import matplotlib.pyplot as plt

sys.path.append('/users/odarwish/abacusutils/')

from abacusnbody.data.read_abacus import read_asdf

import numpy as np

from abacusnbody.analysis.tsc import tsc_parallel

import matplotlib.pyplot as plt

from abacusnbody.analysis.power_spectrum import calc_power, calc_pk_from_deltak, get_k_mu_edges
from abacusnbody.analysis import power_spectrum as ps

from scipy.fft import rfftn, irfftn

import astropy

from classy import Class

import yaml

from pathlib import Path


import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.ticker import LogLocator, LogFormatter, AutoMinorLocator
from matplotlib.gridspec import GridSpec
from matplotlib import patheffects
import matplotlib as mpl

# ---- CONFIGURATION ----
# Use golden ratio for figure dimensions
GOLDEN_RATIO = (5**0.5 - 1) / 2
FIG_WIDTH = 5  # inches
FIG_HEIGHT = FIG_WIDTH * GOLDEN_RATIO
DPI = 300

# Check for and configure LaTeX if available (optional but professional)
# Uncomment this if you have LaTeX installed
# plt.rcParams.update({
#     "text.usetex": True,
#     "font.family": "serif",
#     "font.serif": ["Computer Modern Roman"],
# })

# If not using LaTeX, use a clean serif font
# Try to use TeX fonts that are included with matplotlib
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "cm"  # Use Computer Modern math font
})

# Define a modern, colorblind-friendly palette with higher contrast
# Based on colorblindness-friendly scientific palettes like viridis
# and ones recommended by Nature and Science publications
COLORBLIND_PALETTE = [
    '#0072B2',  # Blue
    '#D55E00',  # Orange
    '#009E73',  # Green
    '#CC79A7',  # Pink
    '#56B4E9',  # Light blue
    '#E69F00',  # Yellow
    '#000000',  # Black
    '#F0E442'   # Light yellow
]

In [2]:
import sympy as sp
import numpy as np
import itertools
import matplotlib.pyplot as plt


class expression():
    # args here is just the list of variables used
    def __init__(self, *args):
        var_list = []
        var_list_names = []
        ns = {}
        for a in args:
            symb = sp.symbols(a)
            setattr(self, a, symb)
            ns[a] = symb
            var_list += [symb]
            var_list_names += [a]
        self.vars = var_list
        self.vars_names = var_list_names
        self.ns = ns

    def add_extra_var(self, name, expression):
        setattr(self, name, expression)

    # similar to add_extra_var but logic of usage is different
    def add_expression(self, expression_name: str, expr):
        setattr(self, expression_name, sp.sympify(expr, locals = self.ns))

    def __add__(self, other_object_expression):
        return

    def get_expression(self, expression_name):
        return getattr(self, expression_name)

    def evaluate_expression(self, expression_name, **valuesdict):
        expr = getattr(self, expression_name)
        expr_numpy = sp.lambdify(self.vars, expr, 'numpy')
        return expr_numpy(**valuesdict)

    def derivate(self, expression_name, variable_of_derivation, save_derivative = False):
        derivative = sp.diff(getattr(self, expression_name), variable_of_derivation)
        if save_derivative:
            der_expression_name = 'der'+str(variable_of_derivation)+'_'+expression_name
            setattr(self, der_expression_name, derivative)
        return derivative

    def change_expression(self, expression_name, new_expression_value):
        setattr(self, expression_name, sp.sympify(new_expression_value, locals = self.ns))
        return


biases_definitions =  {
          'g': 'b10+21/17*b20',
          's': 'b10',
          't': 'b10+7/2*bs2',
          #'phiphi': 'fnl*b10',
          #'c01': 'fnl*2*deltac*(b10-1)',
          #'c11': 'fnl*(2./a1)*(deltac*(b20-2*(a1+a2)*(b10-1.))-a1**2.*(b10-1.))+2.*fnl*deltac*(b10-1.)',
          #'c02': 'fnl**2*4*deltac*((deltac/a1**2.)*(b20-2.*(a1+a2)*(b10-1.))-2.*(b10-1.))'
          }

new_bias_expr = 'b10*(g+s*Ngg/Ngs+t*Ngg/Ngt+e)'
noise_prefix = "N"

variables_list = ['b10', 'b01', 'b11', 'b20', 'bs2', 'fnl', 'nhalo', 'Plin', 'M', 'deltac', 'a1', 'a2', 'Ngg', 'Ngt', 'Nphiphig', 'Ngs', 'Nc02g', 'Nc01g', 'Nc11g', 'new_bias', 'sh_bis', 'sh_tris']


variables_list = ['b10', 'Plin', 'nhalo', 'b20', 'bs2', 'Ngg', 'Ngt', 'Ngs', 'e'] #, '\epsilon']

cov_dict = {
          'Pgg': 'b10**2*Plin+1/nhalo',
          'Pgn': 'new_bias*b10*Plin+sh_bis',
          'Pnn': 'new_bias**2*Plin+Ngg+sh_tris'
        }

cov_dict = {
          'Pgg': 'b10**2*Plin+1/nhalo',
          'Pgn': 'new_bias*Plin',
          'Pnn': '0',
        }


class F(expression):

  def __init__(self, K, *args):
    super().__init__(*args)
    self.length_K = len(K)
    self.K = K

  def add_cov_matrix(self, covariance_matrix_dict, ):
          """Take covariance matrix from input file and store in convenient form.

          Parameters
          ----------
          covariance_matrix_dict : dictionary
              Dictionary defining auto and cross spectra that make up covariance
              matrix (e.g. {'Pgg' : ..., 'Pgn' : ..., 'Pnn' : ...}). Must have
              N(N+1)/2 keys for integer N.
          """
          elements = []
          expressions_list = []
          # Store expressions from covariance_dict
          for key, value in covariance_matrix_dict.items():
              self.add_expression(key, value)
              expressions_list += [self.get_expression(key)]
              elements += [key]
          # Store names of expressions
          self.covmatrix_str = elements

          # Assuming dict has N(N+1)/2 elements, extract N and make NxN matrix
          p = len(self.covmatrix_str)
          matrix_dim = int((-1+np.sqrt(1+8*p))/2)
          self.matrix_dim = matrix_dim

          shape = [matrix_dim, matrix_dim]
          covariance = sp.zeros(*shape)

          # Fill in covariance matrix elements
          for i in range(matrix_dim):
              covariance[i, :] = [expressions_list[i:matrix_dim+i]]
          self.cov_matrix = covariance

  def get_fisher_matrix(self, variables_list = [], var_values = None, verbose = True):
        """Get per-k Fisher matrix, as a matrix of numpy functions.
          Symbolic form deprecated for now.
        Parameters
        ----------
        variables_list : list, optional
            List of variable names to include in Fisher matrix. If not specified,
            everything in self.vars is used.
        verbose : bool, optional
            Whether to print some status updates (default: True).
        """
        matrix_dim = self.matrix_dim
        shape = [matrix_dim, matrix_dim]

         # Compute derivatives of covariance matrix w.r.t. each parameter of interest
        for variable in self.vars:
            der_cov = sp.diff(self.cov_matrix, variable)
            setattr(self, 'der'+str(variable)+'_matrix', der_cov)

        # Make list of parameters to use
        if variables_list == []:
            lista = self.vars
            s = ' all '
        else:
            lista = variables_list
            s = ' '

        if verbose:
            print('Using'+s+'parameters list:', lista)

        self.fisher_list = lista

        combs = list(itertools.combinations_with_replacement(list(lista), 2))

        N = len(lista)
        shape = [N, N]
        fab = sp.zeros(*shape)

        fab_numpy = np.zeros((self.length_K, N, N))

        if verbose:
            print('Calculating fisher matrix per mode')

        for a, b in combs:
            if verbose:
                print('\t%s, %s' % (a,b))
            dera_cov = self.get_expression('der'+str(a)+'_matrix')
            derb_cov = self.get_expression('der'+str(b)+'_matrix')
            i, j = lista.index(a), lista.index(b)
            f = self.__getF__(self.cov_matrix, dera_cov, derb_cov, var_values)
            fab_numpy[:, i, j] = f
            fab_numpy[:, j, i] = f


        self.fisher = fab_numpy
        self.fisher_numpy = fab_numpy

        if verbose:
            print('Done calculating fisher matrix per mode')

  @staticmethod
  def evaluate_matrix_elementwise(sympy_matrix, var_list, var_values, K_len):
    nrows, ncols = sympy_matrix.shape
    evaluated = np.zeros((nrows, ncols, K_len))

    for i in range(nrows):
        for j in range(ncols):
            expr = sympy_matrix[i, j]
            func = sp.lambdify(var_list, expr, 'numpy')
            result = func(**var_values)

            # If result is scalar, broadcast
            if np.isscalar(result):
                evaluated[i, j, :] = result
            else:
                evaluated[i, j, :] = result

    return evaluated


  def __getF__(self, covariance_matrix, dera_covariance_matrix, derb_covariance_matrix, var_values = None):

        """Compute Fisher matrix element.

        Parameters
        ----------
        covariance_matrix : array
            Symbolic version of covariance matrix.
        dera_covariance_matrix : array
            Symbolic version of derivative covariance matrix w.r.t. parameter a.
        derb_covariance_matrix : array
            Symbolic version of derivative covariance matrix w.r.t. parameter b.
        numpify : bool, optional
            Whether to return numpy-computed Fisher matrix (default: False).
        var_values : dict, optional
            Dictionary of variable values for numerical computation.

        Returns
        -------
        final : array
            Fisher element
        """
        all_vars = self.vars


        #Possible improvements: everything in sympy. Just done in 2020branch
        #Problem: numerical precision --> have to use mpmath --> still have loops

        #this is an auxiliarly variable
        #sometimes you can have some matrix sympy entries that are just scalar and to not vectorize
        #po = sp.symbols('po')
        #if po not in all_vars:
        #    all_vars += [po]
        #    var_values['po'] = np.ones(self.length_K)

        """numpy_covariance_matrix = sp.lambdify(all_vars, covariance_matrix, 'numpy')
        numpy_dera_covariance_matrix = sp.lambdify(all_vars, dera_covariance_matrix, 'numpy') #+po*sp.ones(*covariance_matrix.shape), 'numpy')
        numpy_derb_covariance_matrix = sp.lambdify(all_vars, derb_covariance_matrix, 'numpy') #+po*sp.ones(*covariance_matrix.shape), 'numpy')

        # Evaluate covariance and covariance derivatives at input parameter values
        cov_mat = numpy_covariance_matrix(**var_values)
        dera_cov_mat = numpy_dera_covariance_matrix(**var_values)
        derb_cov_mat = numpy_derb_covariance_matrix(**var_values)"""

        cov_mat = self.evaluate_matrix_elementwise(covariance_matrix, all_vars, var_values, self.length_K)
        dera_cov_mat = self.evaluate_matrix_elementwise(dera_covariance_matrix, all_vars, var_values, self.length_K)
        derb_cov_mat = self.evaluate_matrix_elementwise(derb_covariance_matrix, all_vars, var_values, self.length_K)

        shape = cov_mat.shape

        final = []

        # Compute Fisher matrix
        for i in range(shape[-1]):
            cov = cov_mat[:, :, i]
            dera_cov = dera_cov_mat[:, :, i]#-var_values['po'][i]*1
            derb_cov = derb_cov_mat[:, :, i]#-var_values['po'][i]*1
            invC = np.linalg.inv(cov)
            prod = dera_cov@invC@derb_cov@invC
            final += [0.5*np.matrix.trace(prod)]

        final = np.array(final)

        return final



In [24]:
biases_definitions =  {
          'g': 'b10+21/17*b20',
          's': 'b10',
          't': 'b10+7/2*bs2',
          'epsA': 'b10*e*17/6*brB',
          'epsB': 'b10*e*17/6*brA',
          #'phiphi': 'fnl*b10',
          #'c01': 'fnl*2*deltac*(b10-1)',
          #'c11': 'fnl*(2./a1)*(deltac*(b20-2*(a1+a2)*(b10-1.))-a1**2.*(b10-1.))+2.*fnl*deltac*(b10-1.)',
          #'c02': 'fnl**2*4*deltac*((deltac/a1**2.)*(b20-2.*(a1+a2)*(b10-1.))-2.*(b10-1.))'
          }

new_bias_expr = 'b10*(g*Nnn/Ngn+s*Nnn/Nsn+t*Nnn/Ntn+epsA*Nnn/Nx1n+epsB*Nnn/Nx2n)'
noise_prefix = "N"

variables_list = ['b10', 'b01', 'b11', 'b20', 'bs2', 'fnl', 'nhalo', 'Plin', 'M', 'deltac', 'a1', 'a2', 'Ngg', 'Ngt', 'Nphiphig', 'Ngs', 'Nc02g', 'Nc01g', 'Nc11g', 'new_bias', 'sh_bis', 'sh_tris']


variables_list = ['b10', 'Plin', 'nhalo', 'b20', 'bs2', 'Ngn', 'Nsn', 'Ntn', 'e', 'brA', 'brB', 'Nnn', 'Nx2n', 'Nx1n', 'sh_bis'] #, '\epsilon']

variables_list = ['b10', 'Plin']

cov_dict = {
          'Pgg': 'b10**2*Plin+1/nhalo',
          'Pgn': 'new_bias*b10*Plin+sh_bis',
          'Pnn': 'new_bias**2*Plin+Ngg+sh_tris'
        }

cov_dict = {
          'Pgg': 'b10**2*Plin+1/nhalo',
          'Pgn': 'b10*new_bias*Plin+sh_bis',
          'Pnn': '0',
        }

In [25]:
configuration = 'config_abacus.yaml'

with open(configuration, 'r') as f:
    config = yaml.safe_load(f)

output_config = config['output']
filename_prefix = output_config['filename_prefix']
filename_prefix = output_config['filename_prefix']
output_dir = Path(output_config['directory'])/config['name']

gen_power = np.loadtxt(config['power_spectrum']['linear'])
gen_nl_power = np.loadtxt(config['power_spectrum']['nonlinear'])

import jax.numpy as jnp

pnlinf = lambda kmag: jnp.interp(kmag, gen_nl_power[:,0], gen_nl_power[:,1])
gen_power = np.loadtxt(config['power_spectrum']['linear'])
plinf = lambda kmag: jnp.interp(kmag, gen_power[:,0], gen_power[:,1])

kmin, kmax = config['k_range']['kmin'], config['k_range']['kmax']


output_config = config['output']
filename_prefix = output_config['filename_prefix']
filename_prefix = output_config['filename_prefix']
output_dir = Path(output_config['directory'])/config['name']
out_normalization_AB = np.load(output_dir / f"{filename_prefix}_normalization_AB.npy", allow_pickle = True).item()
out_variance_AB = np.load(output_dir / f"{filename_prefix}_variance_AB.npy", allow_pickle = True).item()
analysis_cross_shot_AB = np.load(output_dir / f"{filename_prefix}_cross_shot_AB.npy", allow_pickle = True).item()


kr_config = config['k_range']
kmin = kr_config['kmin']
kmax = kr_config['kmax']
k_samples = kr_config['k_samples']
k_min_analysis = kr_config['k_min_analysis']
k_max_analysis = kr_config['k_max_analysis']

Ks = np.linspace(k_min_analysis, k_max_analysis, k_samples)

from interpax import Interpolator1D

gen_power = np.loadtxt(config['power_spectrum']['linear'])
plinf_jax = Interpolator1D(gen_power[:,0], gen_power[:,1], method="cubic")

In [26]:
b10_A = 2.0

In [27]:
#Ks = np.linspace(0.001, 0.05, 100)
#Ks = out_original["Ks"]
def get_forecast(cov_dict, variables_list_fisher = ["b10"]):

  exp = F(Ks, *variables_list)

  terms = biases_definitions.keys()
  combs = list(itertools.combinations_with_replacement(list(terms), 2))

  for x in terms:
      exec(x+'=0')
      globals()[x] = sp.sympify(biases_definitions[x], locals = exp.ns)
      exp.ns[x] = globals()[x]

  for x, y in combs:
      exec(noise_prefix+x+y+'=0')
      globals()[noise_prefix+x+y] = sp.symbols(noise_prefix+x+y)
      exec(noise_prefix+y+x+'=0')
      globals()[noise_prefix+y+x] = sp.symbols(noise_prefix+y+x)

      exp.ns[noise_prefix+x+y] = globals()[noise_prefix+x+y]
      exp.ns[noise_prefix+y+x] = globals()[noise_prefix+y+x]

  #new_bias = sp.sympify(new_bias_expr, locals = exp.ns)
  exp.new_bias = sp.sympify(new_bias_expr, locals = exp.ns)
  exp.ns['new_bias'] = sp.sympify(new_bias_expr, locals = exp.ns)

  exp.add_cov_matrix(cov_dict)
  exp.cov_matrix


  var_values = {}
  var_values["b10"] = b10_A
  #var_values["b20"] = b20_A
  #var_values["bs2"] = bs2_A
  var_values["Plin"] = plinf(Ks)
  #var_values["nhalo"] = nbar_A
  """
  for keyp in keypairs:
    a, b = keyp
    k = f"N{a}{b}"
    if k in variables_list:
      var_values[k] = out[keyp]**-1.
  """
  #var_values["e"] = 2
  #var_values["brA"] = 3
  #var_values["brB"] = 2
  #var_values["sh_bis"] = out_cross_shot[("n", "n")]*out[("n", "n")]**-1
  #var_values["\epsilon"] = 0

  lista = list(var_values.keys())

  #variables_list_fisher = ["b10"]#, "e"]#, "brA", "brB"]

  all_vars = exp.vars
  temp_cov = exp.evaluate_matrix_elementwise(exp.cov_matrix, all_vars, var_values, exp.length_K)

  exp.get_fisher_matrix(variables_list_fisher, var_values = var_values)
  fisher_numpy = np.nan_to_num(exp.fisher_numpy)

  import scipy

  volume = 1
  volume *= 10**9

  priors = {'b10': '', 'fnl': '', 'b20': '', 'bs2': ''}
  inv_priors = {} #1/val^2, or 0

  f_int = np.zeros((Ks.size, fisher_numpy.shape[1], fisher_numpy.shape[2]))
  for index, Kmin in enumerate(Ks):
    for i in range(fisher_numpy.shape[-1]):
      for j in range(fisher_numpy.shape[-1]):
        FisherPerMode = fisher_numpy[:, i, j]
        function = scipy.interpolate.interp1d(Ks, FisherPerMode)
        result = scipy.integrate.quad(lambda x: function(x)*x**2., Kmin, Ks.max(), epsrel = 1e-15)
        result = result[0]*volume/(4.*np.pi**2.)
        a = variables_list_fisher[i]
        f_int[index, i, j] = result#+inv_priors[a]*int(i==j)

    #self.inv_priors[key] = 1./value**2.

  f_inv = np.linalg.inv(f_int[:-1, :, :])
  #ind = 0
  #errs = f_inv[:, ind, ind]**0.5
  return f_int**(-0.5), f_inv**0.5, temp_cov

In [28]:
gen_power.shape

(2000, 2)

In [31]:
cov_dict = {
          'Pgg': 'b10**2*Plin',#+1/nhalo',
          #'Pgn': 'b10*new_bias*Plin+sh_bis',
          #'Pnn': '0',
        }

errs_un, errs, temp_cov_A = get_forecast(cov_dict, variables_list_fisher = ["b10"])
ind = 0

Using parameters list: ['b10']
Calculating fisher matrix per mode
	b10, b10
Done calculating fisher matrix per mode


/tmp/ipykernel_266591/939951227.py:85: RuntimeWarning: divide by zero encountered in power
  return f_int**(-0.5), f_inv**0.5, temp_cov


In [44]:
volume = 1
volume *= 10**9

Nmodes = volume/(6*np.pi**2)*(Ks[-1]**3-Ks[0]**3)

In [52]:
errs_un[0], errs[0], 1/np.sqrt(2*Nmodes)*b10_A

(array([[0.00235046]]), array([[0.00235046]]), 0.001662028375948665)

In [22]:

cov_dict = {
          'Pgg': 'b10**2*Plin',#+1/nhalo',
          #'Pgn': 'b10*new_bias*Plin+sh_bis',
          #'Pnn': '0',
        }

errs_un, errs, temp_cov_A = get_forecast(cov_dict, variables_list_fisher = ["b10"])
ind = 0
#p = plt.loglog(Ks[:-1], errs[:, ind, ind])
#plt.loglog(Ks, errs_un[:, ind, ind], ls = ":", color = p[0].get_color())


cov_dict = {
          'Pgg': 'b10**2*Plin+1/nhalo',
          #'Pgn': 'b10*new_bias*Plin+sh_bis',
          #'Pnn': '0',
        }

errs_un, errs, temp_cov_B = get_forecast(cov_dict, variables_list_fisher = ["b10"])
ind = 0
#p = plt.loglog(Ks[:-1], errs[:, ind, ind])
#plt.loglog(Ks, errs_un[:, ind, ind], ls = ":", color = p[0].get_color())


cov_dict = {
          'Pgg': 'b10**2*Plin+1/nhalo',
          'Pgn': 'b10*new_bias*Plin+sh_bis',
          'Pnn': '0',
        }

errs_un, errs, temp_cov_C = get_forecast(cov_dict, variables_list_fisher = ["b10", "e"])
ind = 0
#p = plt.loglog(Ks[:-1], errs[:, ind, ind])
#plt.loglog(Ks, errs_un[:, ind, ind], ls = ":", color = p[0].get_color())

ind = 1
p = plt.loglog(Ks[:-1], errs[:, ind, ind], ls = "--")
plt.loglog(Ks, errs_un[:, ind, ind], ls = "-.", color = p[0].get_color())



cov_dict = {
          'Pgg': 'b10**2*Plin', #+1/nhalo',
          #'Pgn': 'b10*new_bias*Plin',
          #'Pnn': 'new_bias**2*Plin',
        }

errs_un, errs, temp_cov_D = get_forecast(cov_dict, variables_list_fisher = ["b10"])
ind = 0
#p = plt.loglog(Ks[:-1], errs[:, ind, ind])
#plt.loglog(Ks, errs_un[:, ind, ind], ls = ":", color = p[0].get_color())

ind = 1
p = plt.loglog(Ks[:-1], errs[:, ind, ind], ls = "--")
plt.loglog(Ks, errs_un[:, ind, ind], ls = "-.", color = p[0].get_color())

plt.ylabel("$\sigma$")
plt.xlabel("$K_{\mathrm{min}}$")

<>:57: SyntaxWarning: invalid escape sequence '\s'
<>:58: SyntaxWarning: invalid escape sequence '\m'
<>:57: SyntaxWarning: invalid escape sequence '\s'
<>:58: SyntaxWarning: invalid escape sequence '\m'
/tmp/ipykernel_266591/2145460989.py:57: SyntaxWarning: invalid escape sequence '\s'
  plt.ylabel("$\sigma$")
/tmp/ipykernel_266591/2145460989.py:58: SyntaxWarning: invalid escape sequence '\m'
  plt.xlabel("$K_{\mathrm{min}}$")
/tmp/ipykernel_266591/2145460989.py:57: SyntaxWarning: invalid escape sequence '\s'
  plt.ylabel("$\sigma$")
/tmp/ipykernel_266591/2145460989.py:58: SyntaxWarning: invalid escape sequence '\m'
  plt.xlabel("$K_{\mathrm{min}}$")


TypeError: _lambdifygenerated() missing 13 required positional arguments: 'nhalo', 'b20', 'bs2', 'Ngn', 'Nsn', 'Ntn', 'e', 'brA', 'brB', 'Nnn', 'Nx2n', 'Nx1n', and 'sh_bis'